# Zero-shot VLM: Qwen2.5-VL (7B)

Runs zero-shot inference for Qwen2.5-VL and writes standardized metrics.

Outputs:
- `results/qwen2_5_vl_zeroshot/metrics.json`
- `results/qwen2_5_vl_zeroshot/predictions.jsonl`


In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None


/workspace/vqa-rag/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# Resolve dataset root


def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").exists():
            return (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").resolve()
        if (p / "0_dataset_prep").exists() and (p / "1_dataset_analysis").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not locate Kvasir_VQA_x1 root. Set KVASIR_VQA_X1_ROOT.")

ROOT = find_kvasir_x1_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.metrics import normalize_answer, compute_metrics

MANIFEST = ROOT / "0_dataset_prep" / "out" / "manifest_x1.parquet"
RESULTS_BASE = ROOT / "2_modeling" / "10_modern_vlm" / "results"

print("ROOT:", ROOT)
print("MANIFEST:", MANIFEST)
print("RESULTS_BASE:", RESULTS_BASE)


ROOT: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
MANIFEST: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/manifest_x1.parquet
RESULTS_BASE: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/10_modern_vlm/results


In [4]:
# Config
QWEN_MODEL_ID = os.getenv("QWEN2_5_VL_MODEL_ID", "Qwen/Qwen2.5-VL-7B-Instruct")
REQUIRE_CUDA = os.getenv("REQUIRE_CUDA", "1") == "1"

MODEL_NAME = "qwen2_5_vl_zeroshot"

SPLITS = ["test"]  # adjust if you want train/validation
MAX_SAMPLES_PER_SPLIT = int(os.getenv("MAX_SAMPLES_PER_SPLIT", "0")) or None

# H200-friendly defaults (override via env vars)
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "8"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "16"))
USE_4BIT = os.getenv("USE_4BIT", "0") == "1"
USE_8BIT = os.getenv("USE_8BIT", "0") == "1"
NUM_IMAGE_WORKERS = int(os.getenv("NUM_IMAGE_WORKERS", str(min(8, os.cpu_count() or 8))))

GEN_KWARGS = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
}

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


In [5]:
# Load manifest
manifest = pd.read_parquet(MANIFEST)
if MAX_SAMPLES_PER_SPLIT:
    manifest = (
        manifest.groupby("split", group_keys=False)
        .head(MAX_SAMPLES_PER_SPLIT)
        .reset_index(drop=True)
    )

if SPLITS:
    manifest = manifest[manifest["split"].isin(SPLITS)].reset_index(drop=True)

print("rows:", len(manifest))

# Resolve image paths if needed
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / "Prototyping_reformat").exists():
            return p
        p = p.parent
    return start.resolve()

REPO_ROOT = find_repo_root(ROOT)

def resolve_image_path(p):
    if p is None:
        return None
    s = str(p)
    path = Path(s)
    if path.exists():
        return str(path)
    # If path contains repo-relative segment, re-root it
    if "/Prototyping_reformat/" in s:
        suffix = s.split("/Prototyping_reformat/", 1)[1]
        cand = REPO_ROOT / "Prototyping_reformat" / suffix
        if cand.exists():
            return str(cand)
    # If stored as repo-relative
    cand = REPO_ROOT / s
    if cand.exists():
        return str(cand)
    # Try common bases
    candidates = [
        ROOT / s,
        ROOT / "0_dataset_prep" / s,
        ROOT / "0_dataset_prep" / "out" / "images" / "all" / Path(s).name,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return str(path)

if "image_abs_path" in manifest.columns:
    manifest["image_abs_path"] = manifest["image_abs_path"].apply(resolve_image_path)
else:
    manifest["image_abs_path"] = manifest["image_path"].apply(resolve_image_path)

missing = manifest[~manifest["image_abs_path"].apply(lambda x: Path(str(x)).exists())]
print("missing resolved images:", len(missing))
if len(missing) > 0:
    print(missing[["image_path", "image_abs_path"]].head())
print("splits:", manifest["split"].value_counts().to_dict())


rows: 15955
missing resolved images: 0
splits: {'test': 15955}


In [6]:
# Helpers
from concurrent.futures import ThreadPoolExecutor


def format_prompt(question: str, processor) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, add_generation_prompt=True)
    return f"User: <image>\nQuestion: {q}\nAssistant:"


def postprocess(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()
    # strip chat prefixes
    for prefix in ["assistant:", "assistant", "answer:"]:
        if t.lower().startswith(prefix):
            t = t[len(prefix):].strip()
    return t


STRIP_CHARS = "'\""

def ensure_list(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        inner = s[1:-1].strip()
        if not inner:
            return []
        if "," in inner:
            parts = [p.strip() for p in inner.split(",")]
        else:
            parts = [p.strip() for p in inner.split()]
        return [p.strip(STRIP_CHARS) for p in parts if p.strip(STRIP_CHARS)]
    return [s]


def _load_image(path: str):
    return Image.open(path).convert("RGB")


def load_images_parallel(paths, max_workers: int):
    if max_workers <= 1:
        return [_load_image(p) for p in paths]
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(_load_image, paths))


In [7]:
def load_model(model_id: str, require_cuda: bool = True):
    if require_cuda and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA not available. Ensure you're using the GPU kernel/environment. "
            "You can set REQUIRE_CUDA=0 to allow CPU, but it will be very slow."
        )

    if (USE_4BIT or USE_8BIT) and BitsAndBytesConfig is None:
        print("Warning: bitsandbytes not installed; disabling 4/8-bit quantization.")
        use_4bit = False
        use_8bit = False
    else:
        use_4bit = USE_4BIT
        use_8bit = USE_8BIT

    quant_config = None
    if torch.cuda.is_available() and BitsAndBytesConfig is not None:
        if use_4bit:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
            )
        elif use_8bit:
            quant_config = BitsAndBytesConfig(load_in_8bit=True)

    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    print("model_type:", getattr(config, "model_type", type(config)))

    device_map = {"": 0} if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        dtype = torch.float32

    # Heuristic: treat configs with vision as vision-to-text
    is_vision = getattr(config, "vision_config", None) is not None
    if getattr(config, "model_type", "") in {"qwen2_5_vl", "qwen2_vl", "llava", "idefics2", "idefics3", "fuyu", "blip_2", "git"}:
        is_vision = True

    if is_vision:
        if AutoModelForVision2Seq is None:
            raise RuntimeError(
                "AutoModelForVision2Seq is not available in your transformers version. "
                "Please upgrade: pip install -U transformers"
            )
        model = AutoModelForVision2Seq.from_pretrained(
            model_id,
            device_map=device_map,
            torch_dtype=dtype,
            quantization_config=quant_config,
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map=device_map,
            torch_dtype=dtype,
            quantization_config=quant_config,
            trust_remote_code=True,
        )

    model.eval()
    if require_cuda:
        first_param = next(model.parameters())
        if first_param.device.type != "cuda":
            raise RuntimeError("Model not on CUDA despite REQUIRE_CUDA=1")
    return processor, model


In [8]:
def run_inference(model_name: str, model_id: str, df: pd.DataFrame):
    out_dir = RESULTS_BASE / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Loading model:", model_id)
    processor, model = load_model(model_id, require_cuda=REQUIRE_CUDA)

    preds = []
    for start in tqdm(range(0, len(df), BATCH_SIZE), desc=f"{model_name} gen"):
        batch = df.iloc[start : start + BATCH_SIZE]
        image_paths = batch["image_abs_path"].tolist()
        images = load_images_parallel(image_paths, NUM_IMAGE_WORKERS)
        prompts = [format_prompt(q, processor) for q in batch["question"].tolist()]

        inputs = processor(images=images, text=prompts, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device, non_blocking=True) for k, v in inputs.items()}

        with torch.inference_mode():
            out = model.generate(**inputs, **GEN_KWARGS)
        decoded = processor.batch_decode(out, skip_special_tokens=True)
        preds.extend([postprocess(t) for t in decoded])

    out_df = df.copy()
    out_df["pred_raw"] = preds
    out_df["pred_norm"] = out_df["pred_raw"].apply(normalize_answer)
    out_df["question_class_list"] = out_df["question_class_list"].apply(ensure_list)

    # Save predictions
    pred_path = out_dir / "predictions.jsonl"
    out_df.to_json(pred_path, orient="records", lines=True)
    print("Wrote", pred_path)

    # Compute metrics
    metrics = {"overall": compute_metrics(out_df["pred_norm"], out_df["answer"]) }

    # By class
    by_class = out_df.explode("question_class_list")
    class_rows = []
    for cls, g in by_class.groupby("question_class_list"):
        m = compute_metrics(g["pred_norm"], g["answer"])
        m["question_class"] = cls
        class_rows.append(m)
    metrics["by_question_class"] = class_rows

    # By complexity
    comp_rows = []
    for c, g in out_df.groupby("complexity"):
        m = compute_metrics(g["pred_norm"], g["answer"])
        m["complexity"] = int(c) if pd.notna(c) else c
        comp_rows.append(m)
    metrics["by_complexity"] = comp_rows

    # Original vs transformed
    trans_rows = []
    for tval, g in out_df.groupby("is_transformed"):
        m = compute_metrics(g["pred_norm"], g["answer"])
        m["is_transformed"] = bool(tval)
        trans_rows.append(m)
    metrics["by_transformed"] = trans_rows

    metrics_path = out_dir / "metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print("Wrote", metrics_path)

    # free VRAM
    del model
    torch.cuda.empty_cache()


In [9]:
import sys, torch
print("python:", sys.executable)
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())


python: /workspace/vqa-rag/bin/python
torch: 2.10.0+cu128
torch cuda: 12.8
cuda available: True


In [ ]:
# Run model
run_inference(MODEL_NAME, QWEN_MODEL_ID, manifest)


Loading model: Qwen/Qwen2.5-VL-7B-Instruct


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

qwen2_5_vl_zeroshot gen:   0%|          | 0/1995 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
The following generation flags are not valid and may be ignored: ['temperature'